# 07 Error analysis

`python -m src.errors` sorts every error of the main runs into buckets with keyword rules; this notebook reads them and explains the main failure.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('../.cache/mpl'))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 200)
from src import config
S = pd.read_csv(config.RESULTS / 'experiments.csv')          # mean (and sd) over seeds
BY_SEED = pd.read_csv(config.RESULTS / 'experiments_by_seed.csv')

def table(models, sets, variant='clean', split='template', metric='f1'):
    t = S[S.model.isin(models) & S.set.isin(sets) & (S.variant == variant) & (S.split == split)]
    return t.pivot_table(index='model', columns='set', values=metric).reindex(models).round(3)

In [2]:
counts = pd.read_csv(config.RESULTS / 'error_buckets.csv')
counts

,where,bucket,nb_word_counts,lr_char,bilstm_finetuned,xlmr,afroxlmr
0,Attacked test (all words),Impersonation missed,0,5,3,31,31
1,Attacked test (all words),Obfuscation missed,62,79,68,0,0
2,Attacked test (all words),Other scam missed,0,10,3,0,22
3,Chichewa,Ambiguous,96,6,4,11,6
4,Chichewa,Chichewa fraud missed,75,149,36,16,113
5,Chichewa,Genuine text flagged,63,2,53,90,5
6,Chichewa,Service-like text flagged,59,26,70,90,37
7,Chichewa,Telco service text flagged,42,14,53,59,26
8,Swahili test,Ambiguous,2,0,0,0,0
9,Swahili test,Genuine text flagged,1,0,1,0,0


In [3]:
err = pd.read_csv(config.RESULTS / 'error_examples.csv', keep_default_na=False)
for (where, b), g in err.groupby(['where', 'bucket']):
    display(Markdown(f'**{where} / {b}** ({len(g)} errors over all models)'))
    display(g.drop_duplicates('text').head(2)[['model', 'label', 'prob', 'text']])

**Attacked test (all words) / Impersonation missed** (70 errors over all models)

,model,label,prob,text
672,lr_char,1,0.220,Habari za mchana. Mimi landlord wako hii number yangu ya Tigo. Mbona kimya na siku zinazidi kwenda...?
687,lr_char,1,0.234,Habari za asubuhi. Mimi landlord wako hii number yangu ya Tigo. Mbona kimya na siku zinazidi kwenda...?


**Attacked test (all words) / Obfuscation missed** (209 errors over all models)

,model,label,prob,text
338,nb_word_counts,1,0.0,Hab​аrі za asubuhi. Мi​mi mwe​nуе nуu​mb4 wako h​iі nа​mba уа​ngu ya Vodа​соm. Mb​оnа k1​my4 na siku zinazidi kwenda...?
339,nb_word_counts,1,0.0,Нab​4ri za siku. Мі​mi mwe​nyе nyu​mbа wako h​іi n4​mbа уa​ngu ya h4lo​tеl. Мb​ona ki​mуа na siku zinazidi kwenda...?


**Attacked test (all words) / Other scam missed** (35 errors over all models)

...

,model,label,prob,text
0,nb_word_counts,0,0.816,Chukua maua yako
2,nb_word_counts,0,0.952,"Samahani, sikuweza kupokea simu yako."


**Swahili test / Genuine text flagged** (2 errors over all models)

,model,label,prob,text
1,nb_word_counts,0,0.811,Naomba unitumie namba ya huyo fundi.
699,bilstm_finetuned,0,0.801,Nakuelewa kaka mwaka juzi nilipata vimawe kwenye urethra but God is awesome nilirecover tuwe tunakunywa sana maji wapendwa na tusibanie ...


**Swahili test / Impersonation missed** (70 errors over all models)

,model,label,prob,text
400,lr_char,1,0.465,Habari za mchana. Mimi mwenye nyumba wako hii namba yangu ya Tigo. Mbona kimya na siku zinazidi kwenda...?
401,lr_char,1,0.482,Habari za asubuhi. Mimi mwenye nyumba wako hii namba yangu ya Tigo. Mbona kimya na siku zinazidi kwenda...?


## The landlord template: why the neural models miss it

31 of the 81 test scams are one script: *"Habari ... Mimi mwenye nyumba wako hii namba yangu ya Vodacom. Mbona kimya ..."* ("Hello, I am your landlord, this is my new number. Why the silence?"). No phone number, no amount, no link: the money request comes later in the conversation. Occlusion shows what each model relies on.

In [4]:
import joblib
from src.classical import scam_proba
from src.perturb import word_effects
lr = joblib.load(config.MODELS / 'lr_char__template.joblib')
nb = joblib.load(config.MODELS / 'nb_word_counts__template.joblib')
landlord = 'Habari za asubuhi. Mimi mwenye nyumba wako hii namba yangu ya Vodacom. Mbona kimya na siku zinazidi kwenda...?'
classic = 'Iyo pesa itume kwenye namba hii <PHONE> jina litakuja JUMA ALLY'
for name, m in [('char LR', lr), ('word NB', nb)]:
    f = lambda t, m=m: scam_proba(m, t)
    for text in (landlord, classic):
        toks = text.split(' ')
        eff = word_effects(text, f)[:4]
        print(f'{name}: P(scam)={f([text])[0]:.2f}', [(toks[i], round(e, 2)) for i, e in eff])

char LR: P(scam)=0.61 [('Vodacom.', 0.21), ('namba', 0.15), ('mwenye', 0.15), ('Mimi', 0.11)]
char LR: P(scam)=1.00 [('namba', 0.0), ('jina', 0.0), ('kwenye', 0.0), ('Iyo', 0.0)]
word NB: P(scam)=1.00 [('Vodacom.', 0.0), ('namba', 0.0), ('Mbona', 0.0), ('mwenye', 0.0)]
word NB: P(scam)=1.00 [('Iyo', 0.0), ('pesa', 0.0), ('itume', 0.0), ('kwenye', 0.0)]


In [5]:
path = config.MODELS / 'afroxlmr__clean__template__s42'
if path.exists():
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(path); model = AutoModelForSequenceClassification.from_pretrained(path).eval()
    @torch.no_grad()
    def tf(texts):
        enc = tok(list(texts), padding=True, truncation=True, max_length=128, return_tensors='pt')
        return torch.softmax(model(**enc).logits, -1)[:, 1].numpy()
    for label, text in [('landlord', landlord), ('landlord + number', landlord + ' <PHONE>'),
                        ('classic scam', classic), ('classic scam - number', classic.replace(' <PHONE>', ''))]:
        print(f'AfroXLMR  {label:22s} P(scam) = {tf([text])[0]:.3f}')
else:
    print('Train and save the model first: python -m src.train_transformer --model afroxlmr --seeds 42 --save')

AfroXLMR  landlord               P(scam) = 0.000
AfroXLMR  landlord + number      P(scam) = 0.985
AfroXLMR  classic scam           P(scam) = 1.000
AfroXLMR  classic scam - number  P(scam) = 0.085


**Reading.** The landlord theme is in training: *namba yangu* (my number) occurs in 49 training messages and *mwenye nyumba* (landlord) in 9, all scams. Word NB catches all 31 copies and word LR 29. The plain transformers do not (0-13% caught): their verdicts rest heavily on the `<PHONE>` token, as the outputs above show, and this scam has no number, amount or link. Number-balanced training (E11, notebook 08) helps: the BiLSTM then catches 68-100% of the template and AfroXLMR 6-32%. The money request in this scam arrives later in the conversation, so a single-message classifier has to rely on the social cue alone.